<a href="https://colab.research.google.com/github/linus-ireri/ML-Projects/blob/main/Week_5_Assignment_%7C_Building_Your_First_Supervised_Machine_Learning_Model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
# Week 5 Assignment: Building Your First Supervised Machine Learning Model
# Project: VeritasRAG

%pip -q install datasets pandas scikit-learn

# 1. Problem Statement

VeritasRAG aims to help students and applicants obtain reliable university information by retrieving relevant passages from trusted documents and using them to generate grounded answers.

For this supervised machine learning exercise, I will use the SQuAD question-answering dataset to predict whether a question is answerable from its provided context. This is a binary classification problem.

The experiment demonstrates supervised learning techniques relevant to question-answering systems. It does not directly evaluate VeritasRAG's retrieval quality or generated-answer accuracy.

In [9]:
# 2. Load the Dataset

from datasets import load_dataset
import pandas as pd
import numpy as np

# Load squad_v2 using its canonical repository identifier to resolve the HF URI error
dataset = load_dataset("rajpurkar/squad_v2", split="train")

df = dataset.shuffle(seed=42).select(
    range(min(5000, len(dataset)))
).to_pandas()

print("Dataset shape:", df.shape)
df.head()

README.md:   0%|          | 0.00/8.92k [00:00<?, ?B/s]

squad_v2/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 16.4MB            

squad_v2/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

squad_v2/validation-00000-of-00001.parqu(…): reconstructing file:   0%|          |  0.00B / 1.35MB            

squad_v2/validation-00000-of-00001.parqu(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/130319 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/11873 [00:00<?, ? examples/s]

Dataset shape: (5000, 5)


,id,title,context,question,answers
0,56e0f3907aa994140058e80a,Canon_law,The Roman Catholic Church canon law also inclu...,What term characterizes the intersection of th...,"{'text': ['full union'], 'answer_start': [104]}"
1,571adcf932177014007e9f56,Athanasius_of_Alexandria,Alexandria was the most important trade center...,What was Alexandria known for?,"{'text': ['important trade center'], 'answer_s..."
2,57325b9fe99e3014001e670c,Jehovah%27s_Witnesses,Former members Heather and Gary Botting compar...,How do the leaders of the Jehovah's Witnesses ...,{'text': ['disparaging individual decision-mak...
3,5728d8be4b864d1900164f6b,Estonia,"Historically, the cuisine of Estonia has been ...",What are the most common foods in Estonia?,"{'text': ['black bread, pork, potatoes, and da..."
4,56f6f5e1711bf01900a44898,Classical_music,Many of the instruments used to perform mediev...,What was the medieval flute made from?,"{'text': ['wood'], 'answer_start': [126]}"


In [10]:
# 3. Prepare Features and Target

# Numerical features derived from the question and context.
df["question_length"] = df["question"].fillna("").str.len()
df["context_length"] = df["context"].fillna("").str.len()
df["question_word_count"] = df["question"].fillna("").str.split().str.len()
df["context_word_count"] = df["context"].fillna("").str.split().str.len()

# Target: 1 if the dataset provides at least one answer, otherwise 0.
df["answerable"] = df["answers"].apply(
    lambda answers: int(len(answers.get("text", [])) > 0) if isinstance(answers, dict) else 0
)

features = [
    "question_length",
    "context_length",
    "question_word_count",
    "context_word_count"
]

X = df[features]
y = df["answerable"]

print("Features:", features)
print("Target: answerable")
print("\nTarget distribution:")
print(y.value_counts())

Features: ['question_length', 'context_length', 'question_word_count', 'context_word_count']
Target: answerable

Target distribution:
answerable
1    3277
0    1723
Name: count, dtype: int64


## 4. Identify the Target and Features

**Target/output:** `answerable`, where 1 indicates that the dataset contains an annotated answer and 0 indicates that it does not.

**Features/input variables:**
- `question_length`: number of characters in the question.
- `context_length`: number of characters in the passage.
- `question_word_count`: number of words in the question.
- `context_word_count`: number of words in the passage.

**Problem type:** Classification. The target has two discrete classes, 0 and 1, rather than a continuous numerical value.

These features are intentionally simple numerical baselines. They do not contain the semantic meaning of the question or passage.

In [11]:
# 5. Split the Dataset into Training and Test Sets

from sklearn.model_selection import train_test_split

# Perform a stratified split to ensure both training and testing sets have representation of both classes
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))
print("\nTraining class distribution:\n", y_train.value_counts())

Training samples: 4000
Testing samples: 1000

Training class distribution:
 answerable
1    2622
0    1378
Name: count, dtype: int64


In [12]:
# 6. Select and Train Two Supervised Learning Models

from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

# Model 1: Logistic Regression
logistic_model = make_pipeline(
    StandardScaler(),
    LogisticRegression(max_iter=1000, random_state=42)
)

# Model 2: Random Forest
random_forest_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    class_weight="balanced"
)

# Fit both models on the freshly split dataset containing both target classes
logistic_model.fit(X_train, y_train)
random_forest_model.fit(X_train, y_train)

print("Both models trained successfully.")

Both models trained successfully.


In [13]:
# 7. Make Predictions on the Test Set

logistic_predictions = logistic_model.predict(X_test)
random_forest_predictions = random_forest_model.predict(X_test)

print("Logistic Regression predictions:")
print(logistic_predictions[:20])

print("\nRandom Forest predictions:")
print(random_forest_predictions[:20])

Logistic Regression predictions:
[1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1]

Random Forest predictions:
[1 1 1 1 0 1 1 1 1 0 0 1 1 1 1 1 1 1 1 1]


In [14]:
# 8. Evaluate Both Models

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

def evaluate_model(name, y_true, predictions):
    return {
        "Model": name,
        "Accuracy": accuracy_score(y_true, predictions),
        "Precision": precision_score(
            y_true, predictions, zero_division=0
        ),
        "Recall": recall_score(
            y_true, predictions, zero_division=0
        ),
        "F1 Score": f1_score(
            y_true, predictions, zero_division=0
        )
    }

results = pd.DataFrame([
    evaluate_model(
        "Logistic Regression",
        y_test,
        logistic_predictions
    ),
    evaluate_model(
        "Random Forest",
        y_test,
        random_forest_predictions
    )
])

display(results.round(4))

,Model,Accuracy,Precision,Recall,F1 Score
0,Logistic Regression,0.655,0.6550,1.0000,0.7915
1,Random Forest,0.587,0.6494,0.8031,0.7181


In [15]:
# 9. Detailed Classification Reports

print("LOGISTIC REGRESSION")
print(classification_report(
    y_test,
    logistic_predictions,
    zero_division=0
))

print("\nRANDOM FOREST")
print(classification_report(
    y_test,
    random_forest_predictions,
    zero_division=0
))

LOGISTIC REGRESSION
              precision    recall  f1-score   support

           0       0.00      0.00      0.00       345
           1       0.66      1.00      0.79       655

    accuracy                           0.66      1000
   macro avg       0.33      0.50      0.40      1000
weighted avg       0.43      0.66      0.52      1000


RANDOM FOREST
              precision    recall  f1-score   support

           0       0.32      0.18      0.23       345
           1       0.65      0.80      0.72       655

    accuracy                           0.59      1000
   macro avg       0.49      0.49      0.47      1000
weighted avg       0.54      0.59      0.55      1000



In [16]:
# 10. Confusion Matrices

print("Logistic Regression Confusion Matrix:")
print(confusion_matrix(y_test, logistic_predictions))

print("\nRandom Forest Confusion Matrix:")
print(confusion_matrix(y_test, random_forest_predictions))

Logistic Regression Confusion Matrix:
[[  0 345]
 [  0 655]]

Random Forest Confusion Matrix:
[[ 61 284]
 [129 526]]


In [17]:
# 11. Compare Model Performance

comparison = results.set_index("Model")
print(comparison.round(4))

best_model = results.loc[results["F1 Score"].idxmax()]

print("\nModel with the highest F1 score:", best_model["Model"])
print("F1 Score:", round(best_model["F1 Score"], 4))

                     Accuracy  Precision  Recall  F1 Score
Model                                                     
Logistic Regression     0.655     0.6550  1.0000    0.7915
Random Forest           0.587     0.6494  0.8031    0.7181

Model with the highest F1 score: Logistic Regression
F1 Score: 0.7915


## 12. Discussion and Comparison

Logistic Regression and Random Forest were trained to classify questions as answerable or unanswerable using numerical features extracted from the question and context.

Accuracy measures the proportion of correct predictions. Precision measures how many questions predicted as answerable were actually answerable. Recall measures how many answerable questions the model successfully identified. The F1 score balances precision and recall.

The model comparison table shows the performance of both approaches on the same test set. The model with the higher F1 score performs better according to that metric, although other metrics should also be considered.

Logistic Regression provides a relatively simple baseline, while Random Forest can learn nonlinear relationships between features. The better-performing model should be selected based on the observed test results rather than assumptions.

The experiment has an important limitation: question and context lengths cannot reliably determine whether a question is semantically answerable. Both models use only numerical text-length features, so their performance does not demonstrate the accuracy of the full VeritasRAG system.

In the production VeritasRAG project, semantic retrieval, relevant document selection, and grounded answer generation would require separate evaluation.

## Conclusion

This assignment demonstrates a supervised machine learning workflow using a question-answering dataset. The target and features were identified, the dataset was divided into training and test sets, two classification models were trained, and their predictions were evaluated using accuracy, precision, recall, and F1 score.

The results provide a baseline for understanding supervised learning in question-answering applications. Further work could incorporate text embeddings and retrieval-related features to better represent semantic relevance, which is central to VeritasRAG.